In [4]:
import ee
import geemap
ee.Authenticate()
ee.Initialize

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

# Create a single DEM image and clip it
elevation = (
    dataset
    .select('DEM')
    .first()
    .clip(panama_geom)
)

# 1000m resolution
smod = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030").select("smod_code")

urban = smod.gte(21).And(smod.lte(30))
rural = smod.gte(11).And(smod.lte(13))

urban_img = urban.unmask(0).toByte()
rural_img = rural.unmask(0).toByte()

distance_urban = urban_img.distance(ee.Kernel.euclidean(50000, "meters")).clip(panama_geom)
distance_rural = rural_img.distance(ee.Kernel.euclidean(50000, "meters")).clip(panama_geom)

# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_urban = (
    distance_urban.resample("bilinear")
    .reproject(collection_projection)
    .rename("urban_reprojected")
    .clip(panama_geom)
)

reprojected_rural = (
    distance_rural.resample("bilinear")
    .reproject(collection_projection)
    .rename("rural_reprojected")
    .clip(panama_geom)
)

Map.addLayer(distance_urban, {"min": 0, "max": 25000}, "Dist Urban")
Map.addLayer(distance_rural, {"min": 0, "max": 25000}, "Dist Rural")
Map.addLayer(reprojected_urban, {"min": 0, "max": 25000}, "Dist Urban Reproject")
Map.addLayer(reprojected_rural, {"min": 0, "max": 25000}, "Dist Rural Reproject")

#Map.addLayer(urban.selfMask(), {"palette": ["red"]}, "Urban")
#Map.addLayer(rural.selfMask(), {"palette": ["blue"]}, "Rural")

Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…